# Logistic Regression Baseline

This notebook trains the **Logistic Regression Baseline** model as part of the Data Mining comparison phase.

It uses the shared model input files generated from:

```text
Power BI ML flat table → Python preprocessing → data/4_model_inputs/
```

The model reuses the shared train/test split and feature definitions.
Preprocessing and classification are combined in one Pipeline.
Preprocessing is fitted on the training split only.

Logistic Regression is the first baseline model.

It uses `class_weight="balanced"` because the target classes are imbalanced.

In [1]:
# Imports

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

import json

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize

In [2]:
# Model import
from sklearn.linear_model import LogisticRegression

In [3]:
# Paths

from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Find the project root whether the notebook is run from the notebook folder or the project root."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "_03_Models").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Open the notebook from inside the project folder."
    )

PROJECT_ROOT = find_project_root(Path.cwd())

MODEL_INPUT_DIR = PROJECT_ROOT / "data" / "4_model_inputs"
RESULTS_DIR = PROJECT_ROOT / "_03_Models" / "results"
SAVED_MODELS_DIR = PROJECT_ROOT / "_03_Models" / "saved_models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Model input folder:", MODEL_INPUT_DIR)
print("Results folder:", RESULTS_DIR)
print("Saved models folder:", SAVED_MODELS_DIR)

Project root: D:\NU\semester_8\data_mining\project\phase I
Model input folder: D:\NU\semester_8\data_mining\project\phase I\data\4_model_inputs
Results folder: D:\NU\semester_8\data_mining\project\phase I\_03_Models\results
Saved models folder: D:\NU\semester_8\data_mining\project\phase I\_03_Models\saved_models


## 1. Load shared model inputs

The shared input files are generated by:

`_02_Data_Cleaning_Preprocessing/02_powerbi_ml_ready_preprocessing.ipynb`

Required files in `data/4_model_inputs/`:

- `train_clean.csv`
- `test_clean.csv`
- `split_metadata.json`

The saved train/test split is reused without splitting the data again.

Categorical encoding and numeric scaling are fitted on the training split as part of the model Pipeline. The fitted Pipeline is then applied to the test split.

In [4]:
required_files = [
    "train_clean.csv",
    "test_clean.csv",
    "split_metadata.json",
]

missing_files = [
    name
    for name in required_files
    if not (MODEL_INPUT_DIR / name).is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing shared model inputs. "
        "Run 02_powerbi_ml_ready_preprocessing.ipynb first.\n"
        f"Missing files: {missing_files}\n"
        f"Expected folder: {MODEL_INPUT_DIR}"
    )

train_df = pd.read_csv(MODEL_INPUT_DIR / "train_clean.csv")
test_df = pd.read_csv(MODEL_INPUT_DIR / "test_clean.csv")

with open(
    MODEL_INPUT_DIR / "split_metadata.json",
    encoding="utf-8",
) as file:
    split_metadata = json.load(file)

TARGET_COLUMN = "TargetEncoded"

for split_name, frame in [
    ("Training", train_df),
    ("Test", test_df),
]:
    if TARGET_COLUMN not in frame.columns:
        raise ValueError(
            f"{split_name} data is missing {TARGET_COLUMN}."
        )

X_train = train_df.drop(columns=TARGET_COLUMN)
X_test = test_df.drop(columns=TARGET_COLUMN)

y_train = train_df[TARGET_COLUMN].astype(int)
y_test = test_df[TARGET_COLUMN].astype(int)

categorical_features = split_metadata["categorical_features"]
numeric_features = split_metadata["numeric_features"]

expected_features = categorical_features + numeric_features

if len(expected_features) != len(set(expected_features)):
    raise ValueError("Feature lists contain duplicate column names.")

for split_name, frame in [
    ("Training", X_train),
    ("Test", X_test),
]:
    if set(frame.columns) != set(expected_features):
        raise ValueError(
            f"{split_name} columns do not match split_metadata.json."
        )

# Keep test columns in the same order as training columns.
X_test = X_test.loc[:, X_train.columns]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("Categorical features:", len(categorical_features))
print("Numeric features:", len(numeric_features))

X_train shape: (3539, 36)
X_test shape: (885, 36)
y_train shape: (3539,)
y_test shape: (885,)
Categorical features: 17
Numeric features: 19


In [5]:
try:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )
except TypeError:
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False,
    )

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            one_hot_encoder,
            categorical_features,
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
    ],
    remainder="drop",
)

## 2. Train the model

In [6]:
MODEL_NAME = "Logistic Regression"

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=42,
            ),
        ),
    ]
)

model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


## 3. Predict on the test set

In [7]:
y_pred = model.predict(X_test)

if hasattr(model, "predict_proba"):
    y_proba = model.predict_proba(X_test)
else:
    y_proba = None

print("Predictions generated.")
print("Probability predictions available:", y_proba is not None)

Predictions generated.
Probability predictions available: True


## 4. Evaluate the model

Accuracy alone is not enough because the dataset is imbalanced.

We report:

- Accuracy
- Balanced Accuracy
- Macro-F1
- Weighted-F1
- Precision, recall, and F1-score per class
- Confusion matrix
- ROC-AUC OvR when probability predictions are available

Dropout recall measures the proportion of actual dropout cases identified by the model.

This experiment includes first- and second-semester academic features, so it does not represent prediction at enrollment time.

In [8]:
target_names = ["Dropout", "Enrolled", "Graduate"]
labels = [0, 1, 2]

accuracy = accuracy_score(y_test, y_pred)
balanced_acc = balanced_accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

if y_proba is not None:
    y_test_binarized = label_binarize(y_test, classes=labels)
    roc_auc_ovr_macro = roc_auc_score(y_test_binarized, y_proba, average="macro", multi_class="ovr")
    roc_auc_ovr_weighted = roc_auc_score(y_test_binarized, y_proba, average="weighted", multi_class="ovr")
else:
    roc_auc_ovr_macro = np.nan
    roc_auc_ovr_weighted = np.nan

report_dict = classification_report(
    y_test,
    y_pred,
    labels=labels,
    target_names=target_names,
    output_dict=True,
    zero_division=0,
)

cm = confusion_matrix(y_test, y_pred, labels=labels)

summary_results = pd.DataFrame([
    {
        "model": MODEL_NAME,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "roc_auc_ovr_macro": roc_auc_ovr_macro,
        "roc_auc_ovr_weighted": roc_auc_ovr_weighted,
        "dropout_precision": report_dict["Dropout"]["precision"],
        "dropout_recall": report_dict["Dropout"]["recall"],
        "dropout_f1": report_dict["Dropout"]["f1-score"],
    }
])

classification_report_df = pd.DataFrame(report_dict).transpose()
confusion_matrix_df = pd.DataFrame(
    cm,
    index=[f"Actual_{name}" for name in target_names],
    columns=[f"Predicted_{name}" for name in target_names],
)

display(summary_results)
display(classification_report_df)
display(confusion_matrix_df)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1,roc_auc_ovr_macro,roc_auc_ovr_weighted,dropout_precision,dropout_recall,dropout_f1
0,Logistic Regression,0.767232,0.73692,0.729085,0.774978,0.893009,0.907999,0.853755,0.760563,0.804469


,precision,recall,f1-score,support
Dropout,0.853755,0.760563,0.804469,284.000000
Enrolled,0.478469,0.628931,0.543478,159.000000
Graduate,0.858156,0.821267,0.839306,442.000000
accuracy,0.767232,0.767232,0.767232,0.767232
macro avg,0.730127,0.736920,0.729085,885.000000
weighted avg,0.788529,0.767232,0.774978,885.000000


,Predicted_Dropout,Predicted_Enrolled,Predicted_Graduate
Actual_Dropout,216,42,26
Actual_Enrolled,25,100,34
Actual_Graduate,12,67,363


## 5. Save results and trained model

In [9]:
summary_results.to_csv(RESULTS_DIR / "logistic_regression_results.csv", index=False)
classification_report_df.to_csv(RESULTS_DIR / "logistic_regression_classification_report.csv")
confusion_matrix_df.to_csv(RESULTS_DIR / "logistic_regression_confusion_matrix.csv")

joblib.dump(model, SAVED_MODELS_DIR / "logistic_regression_model.joblib")

print("Saved:")
print(RESULTS_DIR / "logistic_regression_results.csv")
print(RESULTS_DIR / "logistic_regression_classification_report.csv")
print(RESULTS_DIR / "logistic_regression_confusion_matrix.csv")
print(SAVED_MODELS_DIR / "logistic_regression_model.joblib")

Saved:
D:\NU\semester_8\data_mining\project\phase I\_03_Models\results\logistic_regression_results.csv
D:\NU\semester_8\data_mining\project\phase I\_03_Models\results\logistic_regression_classification_report.csv
D:\NU\semester_8\data_mining\project\phase I\_03_Models\results\logistic_regression_confusion_matrix.csv
D:\NU\semester_8\data_mining\project\phase I\_03_Models\saved_models\logistic_regression_model.joblib


## 6. Interpretation notes

Use these results later in the final comparison notebook.

Focus especially on:

- `balanced_accuracy`
- `macro_f1`
- `weighted_f1`
- `Dropout` recall
- confusion matrix errors between `Dropout`, `Enrolled`, and `Graduate`

Overall accuracy is considered alongside class-level performance.
Higher dropout recall means fewer missed dropout cases, but precision must also be considered because false dropout alerts have a cost.